# 11 · Transition-metal catalysts

Whole complexes rather than the coordination sphere on its own: a catalyst written as SMILES, searched,
gated, and ranked. `07_metal` covers what the sphere can be told to do; this applies it to complete
complexes, with a real ligand set, a conformer search, and a gate deciding what to keep.

In [ ]:
import tempfile

import rxembed as rx
import xyzrender
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")
tmp = tempfile.TemporaryDirectory()

## 1 · A pair of linkage isomers

Square-planar Ni(II) complexes of an O,N/O,C chelate, in pairs: the anionic donor is the amidate N in one
and the carbanion C in the other, same formula and phosphine backbone. Three backbones, so the pairing is
not an artefact of one ligand set.

Both are written with dative bonds (`->[Ni+2]<-`), and each enumerates its own arrangements. `geom_check.check`
is the acceptance test: a sensible M-donor set alone does not catch a wrong periphery.

In [ ]:
PAIRS = {
    "PEt2 diphosphine": (
        "CC[P]1(CC)CC[P](CC)(CC)->[Ni+2]<-12<-[O-]C(=O)C(c1ccccc1)[N-]->2c1ccccc1",
        "CC[P]1(CC)CC[P](CC)(CC)->[Ni+2]<-12<-[O-]C(=O)N(c1ccccc1)[CH-]->2c1ccccc1",
    ),
    "thiosemicarbazone": (
        "C[N]1(C)NC(N)=[S]->[Ni+2]<-12<-[O-]C(=O)C(c1ccccc1)[N-]->2c1ccccc1",
        "C[N]1(C)NC(N)=[S]->[Ni+2]<-12<-[O-]C(=O)N(c1ccccc1)[CH-]->2c1ccccc1",
    ),
    "dppe amidate": (
        "O=C1[O-]->[Ni+2]2(<-[N-](c3ccccc3)C1c1ccccc1)<-[P](CC[P]->2(c1ccccc1)c1ccccc1)(c1ccccc1)c1ccccc1",
        "O=C1[O-]->[Ni+2]2(<-[CH-](c3ccccc3)N1c1ccccc1)<-[P](CC[P]->2(c1ccccc1)c1ccccc1)(c1ccccc1)c1ccccc1",
    ),
}

for name, pair in PAIRS.items():
    for donor, smi in zip(("N-bound", "C-bound"), pair):
        isos = rx.metal(smi, "square_planar")
        isos.summary(details=True)
        for i in isos:
            ens = rx.embed(i, n=1).minimize()
            display(xyzrender.render(ens.mol, stereo=True,hy=True))
            clean = sum(report.ok() for report in ens.check().values())
            print(f"  {name:18s} {donor:8s} {len(isos)} isomer(s)  {ens.n} confs  {clean}/{ens.n} gate-clean")

### Coordinated donor stereochemistry

The same public path handles P, N, and C donors, whether their hand is specified or left for enumeration.
Each selected isomer is embedded, minimized, and required to pass the geometry gate.


In [ ]:
DONOR_STEREO = {
    "P unassigned": "[Pd@SP2+2](<-[Cl-])(<-[Cl-])(<-N)(<-[PH](C)O)",
    "N unassigned": "[Pd@SP2+2](<-[Cl-])(<-[Cl-])(<-N)(<-[NH](C)O)",
    "C unassigned": "[Pd@SP2+2](<-[Cl-])(<-[Cl-])(<-N)(<-[CH-](C)O)",
    "P specified": "[Pd@SP2+2](<-[Cl-])(<-[Cl-])(<-N)(<-[P@H](C)O)",
    "N specified": "[Pd@SP2+2](<-[Cl-])(<-[Cl-])(<-N)(<-[N@H](C)O)",
    "C specified": "[Pd@SP2+2](<-[Cl-])(<-[Cl-])(<-N)(<-[C@H-](C)O)",
}

for name, smi in DONOR_STEREO.items():
    isos = rx.metal(smi, "SPL")
    isos.summary(details=True)
    for iso in isos:
        ens = rx.embed(iso, n=1).minimize()
        display(xyzrender.render(ens.mol, stereo=True, hy=True))
        clean = sum(report.ok() for report in ens.check().values())
        print(f"  {name:12s} {iso.stereo_label:5s} {ens.n} confs  {clean}/{ens.n} gate-clean")

Every donor distance and the coordination shape come back on each pose. The M-donor set is worth reading
back explicitly, since it is what the whole embed was arranged around.


In [ ]:
SMI = PAIRS["PEt2 diphosphine"][0]
iso = rx.metal(SMI, "square_planar")[0]
iso.summary()

ens = rx.embed(iso, n=6).minimize()
seeded = ens.n
ens.filter("geometry")
print(f"geometry gate {ens.n}/{seeded} clean")
for d in iso.donors:
    sym = f"{ens.mol.GetAtomWithIdx(d).GetSymbol()}{d}"
    print(f"  Ni-{sym:5s} {ens.measure((iso.metal, d))['mean']:.2f} A")
xyzrender.render(ens.mol, no_hy=True, bo=False)

## 2 · Searching it

`embed` seeds; `mc` searches. The chain is the same one an organic molecule gets, with
`preset='transition_metal'` so the search knows not to torsion-drive through the coordination sphere.
`prune` then dedups what came back.


In [ ]:
seeded = ens.n  # mc returns the same Ensemble, so read the seed count before searching
searched = ens.mc(preset="transition_metal").minimize()
before = searched.n
searched.prune()
pruned = searched.n
searched.filter("geometry")
print(f"embed {seeded} -> mc {before} -> prune {pruned} -> gate {searched.n} conformers")

xyzrender.render(searched.align().mol, no_hy=True, bo=False)

In [ ]:
searched.landscape()  # kept representatives solid, pruned-away duplicates faded

## 3 · Both hands of a racemate

The amidate backbone has an unspecified stereocentre. `rx.embed(..., metal=)` returns one ensemble per hand,
each carrying its own constraints through the chain. The hands are never pruned against each other; compare
them only after a real-energy calculation.

In [ ]:
racemate = rx.embed(SMI, metal="square_planar", n=4).minimize().prune()

for e in racemate:
    total = e.n
    e.filter("geometry")
    print(f"  stereo {e.tag['stereo']}: {e.n}/{total} gate-clean conformers")

paths = racemate.dump(f"{tmp.name}/catalyst.xyz")  # one tagged .xyz per hand
print("dumped:", [p.split("/")[-1] for p in paths])

In [ ]:
try:
    racemate.best()
except ValueError as e:
    print("best() refused:", e)
# Ranking the hands needs real energies: racemate.score("gxtb").best() (xtb on $XTB_EXE).
# Enantiomers share a formula, so g-xTB totals do compare.

## 4 · A real metal TS: reacting core frozen, spectator ferrocene held

A bimetallic Mn/Fe transition state: H₂ activation at Mn (H–H breaking, Mn–H / substrate N–H, O–H forming),
with a spectator ferrocene. Freeze the reacting core, search the periphery, then check it with the public
frozen-core gate; both metals are restored. `ts_bonds=` labels the real forming/breaking bonds with their
live lengths, including the ~0.86 Å H–H being cleaved.

In [ ]:
mn = "structures/mn-h2.xyz"
rc = [1, 5, 63, 64, 65, 66]  # reacting-core atoms (from a graphrc trajectory in practice)
bonds = [(5, 65), (64, 66), (1, 64), (65, 66), (63, 64)]  # forming / breaking bonds held during embed
mn_mol = rx.read_xyz(mn, charge=-1)  # the bridging hydride carries the anion's formal charge
ens = rx.embed(mn_mol, fix=rc, n=6).mc(preset="ensemble").minimize()
total = ens.n
ens.filter("geometry", frozen=rc, reference=mn)
print(f"{total} conformers | frozen-core gate {ens.n}/{total} clean")
xyzrender.render(
    ens.mol,
    no_hy=True,
    ts_bonds=[(i + 1, j + 1) for i, j in bonds],
    labels=[f"{i + 1} {j + 1} d" for i, j in bonds],
)

### Change fac to mer without moving the transition-state core

The input Mn arrangement is fac/Λ. Freeze the six reacting atoms, enumerate only the free Mn sites, and
select mer/Λ; `center="Mn"` retains the input ferrocene face. CXSMILES stores both metal arrangements;
pass `fix=` again when embedding the selected state.

In [ ]:
input_fac = rx.metal(rx.cxsmiles(mn_mol), center="Mn", stereo="free")[0]
isomers = rx.metal(mn_mol, "OCT", center="Mn", fix=rc, stereo="free")
isomers.summary()
target = isomers.select(label="mer", hand="lambda", index=4)  # donor ordering leaves 3 lambda mer arrangements
written = rx.cxsmiles(target)
print("round-trips to the written CXSMILES:", rx.cxsmiles(rx.metal(written, center="Mn", stereo="free")[0]) == written)
print("Fe free-stereo isomer count:", len(rx.metal(written, center="Fe", stereo="free")))
generated = rx.embed(target, n=4, seed=1)
total = generated.n
generated.filter("geometry", frozen=rc, reference=mn)
print(
    f"{input_fac.label}/{input_fac.chirality} -> {target.label}/{target.chirality} | geometry gate {generated.n}/{total} clean"
)
print(written)
xyzrender.render(generated.mol, no_hy=True, ts_bonds=[(i + 1, j + 1) for i, j in bonds])

## 5 · Seating a substrate, and switching its binding mode

Reach a substrate donor into the vacant site with `coordinate=`. A Pt(amine)Cl₂ has one open site; seat an
acetone carbonyl O there (a coordinative Pt–O bond, drawn dotted + labelled).


In [ ]:
isos = rx.metal("N->[Pt+2](<-[Cl-])<-[Cl-].CC(C)=O", "square_planar")  # 3 donors + acetone -> one vacant site
iso = isos.select(index=0)  # the cis isomer (both Cl cis)
O = next(a.GetIdx() for a in iso.mol.GetAtoms() if a.GetSymbol() == "O")
seated = rx.embed(iso, coordinate=O, n=3).mc(preset="ensemble").minimize().prune()
for report in seated.check().values():
    print(report.summary())
print(f"acetone O seated: Pt-O = {seated.measure((iso.metal, O))['mean']:.2f} Å  ({len(seated)} confs)")
xyzrender.render(seated.mol, no_hy=True, labels=[f"{iso.metal + 1} {O + 1} d"])  # label the new Pt-O coordinative bond

The same substrate can bind the other way instead: hydrogen-bonded to the amine, not seated on the metal.
One `contacts=` grip states it, and `Pt-O` reports which mode you got.

The embed biases a grip toward the window midpoint and restrained UFF keeps it inside the window. `mc` does
not model the contact itself, so `minimize` reapplies the constraint afterwards.

In [ ]:
N = next(d for d in iso.donors if iso.mol.GetAtomWithIdx(d).GetSymbol() == "N")
H = next(n.GetIdx() for n in iso.mol.GetAtomWithIdx(N).GetNeighbors() if n.GetAtomicNum() == 1)
grip = rx.Contact(distances={(min(H, O), max(H, O)): (1.7, 2.3)}, angles={(N, H, O): (150.0, 180.0)}, label="NH..O")

hb = rx.embed(iso, contacts=grip, n=4)  # no coordinate=: the acetone grips the amine, not the metal
print(
    f"seeded  : H-O {hb.measure((H, O))['mean']:.2f} A (window 1.7-2.3), "
    f"Pt-O {hb.measure((iso.metal, O))['mean']:.2f} A"
)
hb = hb.mc(preset="ensemble").minimize().prune()
for report in hb.check().values():
    print(report.summary())
print(
    f"searched: H-O {hb.measure((H, O))['mean']:.2f} A, "
    f"N-H-O {hb.measure((N, H, O))['mean']:.0f} deg, {hb.n} gate-clean"
)

xyzrender.render(hb.mol, no_hy=True, nci_bonds=[(H + 1, O + 1)], labels=[f"{H + 1} {O + 1} d"])